# Config 1 static XAS — per-shot inspection

Load the per-shot (tofs_e, tofs_i, gmd) data produced by
`compute_static_xas_cfg1.py` and plot, for both electron and ion TOF:

1. The GMD-normalised TOF spectrum in each photon-energy bin on its
   own panel (wrapped at `MAX_COLS`).
2. The same spectra stacked into a 2D map (photon energy × TOF).

Normalisation is sum-ratio: `sum(hits in bin) / sum(GMD across shots)`,
matching the project convention (see `bin_and_sum_ratio` in
`analysis/scripts/binning.py`).

TOF values are the raw 100 ps units stored by `write_h5.py`.

In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

%matplotlib inline

# Locate repo root and put analysis/scripts on the path.
cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "analysis" / "scripts").exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError("Could not find analysis/scripts in any parent directory.")
sys.path.insert(0, str(repo_root / "analysis" / "scripts"))

import config as path_config  # noqa: E402

## Parameters

Point `INPUT_H5` at the file produced by `compute_static_xas_cfg1.py`.
TOF binning is independent for electron and ion channels because their
useful ranges differ. All TOF bin edges are in 100 ps (the raw write_h5
unit).

In [ ]:
RUN_NO   = 48346
INPUT_H5 = Path(path_config.COMBINED_DIR).parent / "xas_static" / f"run{RUN_NO}_cfg1_static_xas.h5"

# TOF histogram edges (100 ps units). Adjust to taste — narrower bins
# give finer spectra at the cost of noisier panels.
ETOF_BIN_EDGES = np.linspace(0, 40000, 1001)
ITOF_BIN_EDGES = np.linspace(0, 40000, 1001)

# Subplot wrapping — columns per row in the per-energy panel grids.
MAX_COLS = 4

# Use log colour scale on the 2D maps.
LOG_2D = False

TOF_UNIT_LABEL = "TOF (100 ps)"  # change to 'TOF (ns)' if you rescale below

## Load

In [ ]:
with h5py.File(INPUT_H5, "r") as f:
    tofs_e           = f["tofs_e"][...]            # (N_E, N_shots, max_ecounts)
    tofs_i           = f["tofs_i"][...]            # (N_E, N_shots, max_icounts)
    gmd              = f["gmd"][...]               # (N_E, N_shots) NaN-padded
    n_shots          = f["n_shots"][...]           # (N_E,)
    nominal_energies = f["nominal_energies"][...]  # (N_E,)
    attrs            = dict(f.attrs)

N_E = nominal_energies.size
print(f"loaded {INPUT_H5.name}")
print(f"  n_energies      = {N_E}")
print(f"  tofs_e          = {tofs_e.shape}  (max_ecounts = {tofs_e.shape[-1]})")
print(f"  tofs_i          = {tofs_i.shape}  (max_icounts = {tofs_i.shape[-1]})")
print(f"  shots per E     = min {n_shots.min()}, max {n_shots.max()}, total {int(n_shots.sum())}")
print(f"  energy range    = {nominal_energies.min():.2f} .. {nominal_energies.max():.2f} eV")
print(f"  signal_bunches  = {tuple(attrs.get('signal_bunch_range', []))}")

## Histogram TOFs per energy bin, normalise to total GMD

For each energy section:

* Truncate to the `n_shots[ie]` valid shots (the rest is NaN/zero padding).
* Pool TOF hits from every shot (zero-padded slots are dropped as 0 < bin_edges[0] is forbidden by clipping the lower edge).
* Divide the histogram by `sum(GMD)` over the same shots so the
  resulting spectrum is in *hits per µJ-bin* (the project's sum-ratio
  convention).

In [ ]:
def _energy_normalised_tof_spectra(tofs, gmd, n_shots, edges):
    """
    Per-energy histogram of non-zero TOF hits, divided by sum(GMD).

    Returns
    -------
    spec : (N_E, n_bins) float64
        Hits-per-bin per µJ. NaN where sum(GMD) is zero.
    sum_gmd : (N_E,) float64
        Total GMD per energy section (denominator), µJ.
    n_valid : (N_E,) int64
        Number of valid shots contributing to the histogram.
    """
    N_E = tofs.shape[0]
    n_bins = len(edges) - 1
    spec    = np.zeros((N_E, n_bins), dtype=np.float64)
    sum_gmd = np.zeros(N_E, dtype=np.float64)
    n_valid = np.zeros(N_E, dtype=np.int64)
    for ie in range(N_E):
        n = int(n_shots[ie])
        if n == 0:
            spec[ie] = np.nan
            continue
        G_slice = gmd[ie, :n]
        ok      = np.isfinite(G_slice)
        if not ok.any():
            spec[ie] = np.nan
            continue
        sum_gmd[ie] = float(G_slice[ok].sum())
        n_valid[ie] = int(ok.sum())
        hits = tofs[ie, :n][ok]                # (n_valid, max_hits)
        hits = hits.ravel()
        hits = hits[hits > 0]                  # drop zero-padding
        if hits.size == 0 or sum_gmd[ie] == 0.0:
            continue
        h, _ = np.histogram(hits, bins=edges)
        spec[ie] = h / sum_gmd[ie]
    # Mark zero-denominator rows as NaN so plots flag them.
    spec[sum_gmd == 0.0] = np.nan
    return spec, sum_gmd, n_valid


spec_e, sum_gmd_e, n_valid_e = _energy_normalised_tof_spectra(
    tofs_e, gmd, n_shots, ETOF_BIN_EDGES,
)
spec_i, sum_gmd_i, n_valid_i = _energy_normalised_tof_spectra(
    tofs_i, gmd, n_shots, ITOF_BIN_EDGES,
)

etof_cents = 0.5 * (ETOF_BIN_EDGES[:-1] + ETOF_BIN_EDGES[1:])
itof_cents = 0.5 * (ITOF_BIN_EDGES[:-1] + ITOF_BIN_EDGES[1:])

print(f"spec_e shape = {spec_e.shape}  (E × n_etof_bins)")
print(f"spec_i shape = {spec_i.shape}  (E × n_itof_bins)")
print(f"valid shots per E (eTOF) = min {n_valid_e.min()}, max {n_valid_e.max()}")
print(f"total GMD per E (uJ)     = min {sum_gmd_e.min():.2f}, max {sum_gmd_e.max():.2f}")

## Plot 1 — eTOF spectrum per photon energy

One panel per nominal energy, wrapped at `MAX_COLS`. All panels share
y-limits so peak heights are directly comparable across energies.

In [ ]:
def _panel_grid(N, max_cols, panel_w=3.4, panel_h=2.0):
    cols  = min(max_cols, N)
    rows  = (N + cols - 1) // cols
    fig, axes = plt.subplots(
        rows, cols,
        figsize=(panel_w * cols, panel_h * rows),
        sharex=True, sharey=True, squeeze=False,
    )
    return fig, axes, rows, cols


def _plot_per_energy(spec, cents, energies, *, title, ylabel, xlabel, max_cols):
    N_E = spec.shape[0]
    fig, axes, rows, cols = _panel_grid(N_E, max_cols)
    finite = spec[np.isfinite(spec)]
    ymax = float(np.nanpercentile(finite, 99.5)) if finite.size else 1.0
    if ymax <= 0:
        ymax = float(np.nanmax(spec)) or 1.0
    for k in range(rows * cols):
        r, c = divmod(k, cols)
        ax = axes[r, c]
        if k >= N_E:
            ax.set_visible(False)
            continue
        if np.all(np.isnan(spec[k])):
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes, color="0.4")
        else:
            ax.plot(cents, spec[k], lw=0.8)
        ax.set_ylim(0, 1.05 * ymax)
        ax.set_title(f"{energies[k]:.2f} eV", fontsize=9)
        if r == rows - 1:
            ax.set_xlabel(xlabel)
        if c == 0:
            ax.set_ylabel(ylabel)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    return fig, axes


fig, _ = _plot_per_energy(
    spec_e, etof_cents, nominal_energies,
    title="GMD-normalised eTOF spectrum per photon energy",
    ylabel="hits / uJ / bin",
    xlabel=TOF_UNIT_LABEL,
    max_cols=MAX_COLS,
)

## Plot 2 — iTOF spectrum per photon energy

In [ ]:
fig, _ = _plot_per_energy(
    spec_i, itof_cents, nominal_energies,
    title="GMD-normalised iTOF spectrum per photon energy",
    ylabel="hits / uJ / bin",
    xlabel=TOF_UNIT_LABEL,
    max_cols=MAX_COLS,
)

## Plot 3 — 2D map (photon energy × eTOF)

Each row of the map is the spectrum in the corresponding panel of plot 1.
The energy axis can be non-uniform; `pcolormesh` uses half-step edges so
each row is centred on its nominal energy.

In [ ]:
def _half_step_edges(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 1:
        return np.array([x[0] - 0.5, x[0] + 0.5])
    mid = 0.5 * (x[:-1] + x[1:])
    return np.concatenate([[2 * x[0] - mid[0]], mid, [2 * x[-1] - mid[-1]]])


def _plot_2d_map(spec, cents_edges, energies, *, title, xlabel, log=False):
    e_edges = _half_step_edges(energies)
    fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
    finite = spec[np.isfinite(spec)]
    if log:
        positive = finite[finite > 0]
        vmin = float(np.nanpercentile(positive, 2)) if positive.size else 1.0
        vmax = float(np.nanpercentile(positive, 99.5)) if positive.size else 10.0
        if vmin <= 0:
            vmin = max(1e-12, vmax * 1e-4)
        norm = LogNorm(vmin=vmin, vmax=vmax)
    else:
        vmin = float(np.nanpercentile(finite, 1)) if finite.size else 0.0
        vmax = float(np.nanpercentile(finite, 99.5)) if finite.size else 1.0
        norm = None
    im = ax.pcolormesh(
        cents_edges, e_edges, spec,
        cmap="viridis", shading="auto",
        vmin=None if log else vmin, vmax=None if log else vmax,
        norm=norm,
    )
    fig.colorbar(im, ax=ax, label="hits / uJ / bin")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("nominal photon energy (eV)")
    ax.set_title(title)
    return fig, ax


fig, _ = _plot_2d_map(
    spec_e, ETOF_BIN_EDGES, nominal_energies,
    title="GMD-normalised eTOF vs photon energy",
    xlabel=TOF_UNIT_LABEL,
    log=LOG_2D,
)

## Plot 4 — 2D map (photon energy × iTOF)

In [ ]:
fig, _ = _plot_2d_map(
    spec_i, ITOF_BIN_EDGES, nominal_energies,
    title="GMD-normalised iTOF vs photon energy",
    xlabel=TOF_UNIT_LABEL,
    log=LOG_2D,
)